
# Regresión Lineal Múltiple — Enfoque Estadístico (statsmodels)

Este cuaderno ilustra un flujo de trabajo **estadístico** (similar a `lm` en R) para una regresión lineal múltiple:
- Generación de un dataset sintético (precio de viviendas).
- Ajuste con `statsmodels.OLS`.
- Interpretación con `summary()` (coeficientes, t-test, F-test, R² ajustado, Jarque-Bera, etc.).
- Diagnósticos: residuos vs. ajustados, QQ-plot, prueba de Breusch–Pagan (heterocedasticidad).
- Multicolinealidad: cálculo del **VIF**.
- Intervalos de confianza para parámetros; predicción con **IC** e **IP**.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

np.random.seed(42)

# ----- Dataset simulado: precio de viviendas -----
n = 120
superficie = np.random.randint(45, 220, n)      # m^2
habitaciones = np.random.randint(1, 6, n)       # 1..5
antiguedad = np.random.randint(0, 40, n)        # años

# Ecuación generadora (lineal) + ruido Gaussiano
precio = (
    60000
    + 1300*superficie
    + 18000*habitaciones
    - 1000*antiguedad
    + np.random.normal(0, 25000, n)
)

data = pd.DataFrame({
    "precio": precio,
    "superficie": superficie,
    "habitaciones": habitaciones,
    "antiguedad": antiguedad
})

data.head()


In [ ]:

# --- Gráficos exploratorios simples (matplotlib puro) ---
for col in ["superficie", "habitaciones", "antiguedad"]:
    plt.figure()
    plt.scatter(data[col], data["precio"])
    plt.xlabel(col)
    plt.ylabel("precio")
    plt.title(f"precio vs. {col}")
    plt.show()

# Matriz de correlación (numérica)
data.corr(numeric_only=True)


In [ ]:

# ----- Ajuste con statsmodels (OLS) -----
X = data[["superficie", "habitaciones", "antiguedad"]]
X = sm.add_constant(X)  # agrega intercepto
y = data["precio"]

modelo = sm.OLS(y, X).fit()

# Resumen estilo R
print(modelo.summary())


In [ ]:

# ----- Intervalos de confianza para parámetros (95%) -----
ci_params = modelo.conf_int(alpha=0.05)
ci_params.columns = ["2.5%", "97.5%"]
ci_params


In [ ]:

# ----- Diagnóstico de residuos -----
fitted = modelo.fittedvalues
resid = modelo.resid

# Residuos vs. ajustados
plt.figure()
plt.scatter(fitted, resid)
plt.axhline(0, linestyle="--")
plt.xlabel("Valores ajustados")
plt.ylabel("Residuos")
plt.title("Residuos vs. Ajustados")
plt.show()

# QQ-plot de residuos
sm.qqplot(resid, line="45", fit=True)
plt.title("QQ-plot de residuos")
plt.show()


In [ ]:

# ----- Prueba de Breusch–Pagan -----
# H0: varianza constante (homocedasticidad)
bp_test = het_breuschpagan(resid, X)
bp_labels = ["LM stat", "LM p-value", "F stat", "F p-value"]
dict(zip(bp_labels, bp_test))


In [ ]:

# ----- Multicolinealidad: VIF -----
import numpy as np
vif_df = pd.DataFrame()
vif_df["variable"] = X.columns
vif_df["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_df


In [ ]:

# ----- Predicción con IC e IP -----
nuevo = pd.DataFrame({
    "const": 1.0,
    "superficie": [80, 120, 180],
    "habitaciones": [2, 3, 4],
    "antiguedad": [5, 10, 20]
})

pred_res = modelo.get_prediction(nuevo)
pred_summary = pred_res.summary_frame(alpha=0.05)  # 95%
pred_summary  # mean_ci_* (IC), obs_ci_* (IP)



## Notas finales

- Use `summary()` como en R para inferencia (coeficientes, p-valores, F-test, R² ajustado).
- Revise **residuos vs. ajustados** y **QQ-plot** para validar linealidad/normalidad.
- Use **Breusch–Pagan** para homocedasticidad y **VIF** para multicolinealidad.
- Reporte intervalos de confianza de parámetros, y para predicciones utilice **IC** (media) e **IP** (nueva observación).
